In [1]:
!apt-get update -y && !apt-get install -y supervisor
!pip install Flask

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [75.3 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,067 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.4 MB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,409 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [

In [2]:
import os
import sqlite3

# Create required directories
os.makedirs("templates", exist_ok=True)
os.makedirs("static/css", exist_ok=True)
os.makedirs("static/js", exist_ok=True)

# 1. config.py
with open("config.py", "w") as f:
    f.write('''
DB_NAME = "sensor_history.db"
LOG_FILE = "app_activity.log"
SAMPLING_INTERVAL = 5
STALE_THRESHOLD_SECONDS = 15
TEMP_HIGH_THRESHOLD = 75.0
ALERT_PERSISTENCE_COUNT = 3
ALERT_COOLDOWN_SECONDS = 30
''')

# 2. database.py (Improved Logging & Exception Handling)
with open("database.py", "w") as f:
    f.write('''
import sqlite3
import datetime
import logging
from config import DB_NAME, LOG_FILE

logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

def init_db():
    try:
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS sensor_readings (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                temperature REAL,
                humidity REAL,
                status TEXT CHECK(status IN ('ACCEPTED', 'REJECTED', 'MISSING', 'STALE'))
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS system_logs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                event_type TEXT,
                message TEXT,
                health_state TEXT,
                component TEXT,
                exception_type TEXT
            )
        """)
        conn.commit()
        conn.close()
    except sqlite3.OperationalError as e:
        logging.error(f"[COMPONENT: DATABASE] [PATH: {DB_NAME}] [EXCEPTION: {type(e).__name__}] Database initialization failed: {e}")

def log_event(event_type, message, health_state="HEALTHY", component="MONITOR_ENGINE", exception_type="NONE"):
    try:
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO system_logs (event_type, message, health_state, component, exception_type)
            VALUES (?, ?, ?, ?, ?)
        """, (event_type, message, health_state, component, exception_type))
        conn.commit()
        conn.close()
    except sqlite3.OperationalError as e:
        logging.error(f"[COMPONENT: {component}] [PATH: {DB_NAME}] [EXCEPTION: {type(e).__name__}] Unable to write system log: {e}")

def insert_reading(temp, humidity, status, retry_count=0):
    try:
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        now = datetime.datetime.utcnow().isoformat()
        cursor.execute("""
            INSERT INTO sensor_readings (timestamp, temperature, humidity, status)
            VALUES (?, ?, ?, ?)
        """, (now, temp, humidity, status))
        conn.commit()
        conn.close()
    except sqlite3.OperationalError as e:
        logging.error(f"[COMPONENT: SENSOR_GENERATOR] [PATH: {DB_NAME}] [RETRY: {retry_count}] [EXCEPTION: {type(e).__name__}] Database write error: {e}")
        raise e

def get_latest_reading():
    try:
        conn = sqlite3.connect(DB_NAME)
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM sensor_readings ORDER BY id DESC LIMIT 1")
        row = cursor.fetchone()
        conn.close()
        return dict(row) if row else None
    except Exception:
        return None

def get_historical_readings(limit=50):
    try:
        conn = sqlite3.connect(DB_NAME)
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM sensor_readings ORDER BY id DESC LIMIT ?", (limit,))
        rows = cursor.fetchall()
        conn.close()
        return [dict(r) for r in rows]
    except Exception:
        return []
''')

# 3. sensor_generator.py
with open("sensor_generator.py", "w") as f:
    f.write('''
import time
import random
from database import insert_reading, init_db
from config import SAMPLING_INTERVAL

def run_sensor_generator():
    init_db()
    while True:
        try:
            temperature = round(random.uniform(20.0, 25.0), 2)
            humidity = round(random.uniform(45.0, 55.0), 2)
            insert_reading(temperature, humidity, "ACCEPTED")
        except Exception as e:
            print(f"[SENSOR_GENERATOR ERROR] {e}")
        time.sleep(SAMPLING_INTERVAL)

if __name__ == "__main__":
    run_sensor_generator()
''')

# 4. monitor_engine.py
with open("monitor_engine.py", "w") as f:
    f.write('''
import time
import logging
from datetime import datetime
from config import LOG_FILE, STALE_THRESHOLD_SECONDS, TEMP_HIGH_THRESHOLD, ALERT_PERSISTENCE_COUNT, ALERT_COOLDOWN_SECONDS
from database import get_latest_reading, log_event, insert_reading

logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

class MonitorEngine:
    def __init__(self):
        self.health_state = "RECOVERING"
        self.consecutive_high_temp = 0
        self.last_alert_time = 0
        self.alert_active = False
        log_event("STARTUP", "System startup initiated.", self.health_state, component="MONITOR_ENGINE")
        time.sleep(2)
        self.health_state = "HEALTHY"
        log_event("STATE_CHANGE", "System transitioned to HEALTHY state.", self.health_state, component="MONITOR_ENGINE")

    def evaluate(self):
        latest = get_latest_reading()
        now = datetime.utcnow()
        if not latest:
            self.health_state = "DEGRADED"
            log_event("MISSING_DATA", "No reading available", self.health_state, component="MONITOR_ENGINE")
            return
        last_ts = datetime.fromisoformat(latest['timestamp'])
        time_diff = (now - last_ts).total_seconds()
        if time_diff > STALE_THRESHOLD_SECONDS:
            if self.health_state != "DEGRADED":
                self.health_state = "DEGRADED"
                log_event("STALE_DATA", f"Sensor stale ({time_diff:.1f}s delay)", self.health_state, component="MONITOR_ENGINE")
                try:
                    insert_reading(latest['temperature'], latest['humidity'], 'STALE')
                except Exception:
                    pass
            return
        if self.health_state == "DEGRADED" and time_diff <= STALE_THRESHOLD_SECONDS:
            self.health_state = "HEALTHY"
            log_event("RECOVERY", "Sensor stream resumed.", self.health_state, component="MONITOR_ENGINE")

if __name__ == "__main__":
    engine = MonitorEngine()
    while True:
        engine.evaluate()
        time.sleep(3)
''')

# 5. app.py
with open("app.py", "w") as f:
    f.write('''
from flask import Flask, render_template_string, jsonify
from database import init_db, get_latest_reading, get_historical_readings
import sqlite3
from config import DB_NAME, STALE_THRESHOLD_SECONDS, TEMP_HIGH_THRESHOLD

app = Flask(__name__)
init_db()

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Unattended System Monitor - Assignment 7.2</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
</head>
<body class="bg-light">
<div class="container py-4">
    <h1 class="mb-4">System Diagnostic & Monitoring Dashboard</h1>
    <div class="row mb-4">
        <div class="col-md-3">
            <div class="card text-center p-3 shadow-sm">
                <h5 class="text-muted">Health State</h5>
                <h2 id="health-state" class="fw-bold">LOADING...</h2>
            </div>
        </div>
        <div class="col-md-3">
            <div class="card text-center p-3 shadow-sm">
                <h5 class="text-muted">Current Temp</h5>
                <h2 id="current-temp" class="fw-bold">-- °C</h2>
            </div>
        </div>
        <div class="col-md-3">
            <div class="card text-center p-3 shadow-sm">
                <h5 class="text-muted">Status</h5>
                <h2 id="sensor-status" class="fw-bold">--</h2>
            </div>
        </div>
        <div class="col-md-3">
            <div class="card text-center p-3 shadow-sm">
                <h5 class="text-muted">Latest Update</h5>
                <p id="latest-timestamp" class="small text-muted mt-2">--</p>
            </div>
        </div>
    </div>
</div>
<script>
    async function update() {
        try {
            const res = await fetch('/api/status');
            const data = await res.json();
            if (data.current_reading) {
                document.getElementById('current-temp').innerText = `${data.current_reading.temperature} °C`;
                document.getElementById('sensor-status').innerText = data.current_reading.status;
                document.getElementById('latest-timestamp').innerText = data.current_reading.timestamp;
            }
            const healthElem = document.getElementById('health-state');
            healthElem.innerText = data.health_state;
            healthElem.className = `fw-bold ${data.health_state === 'HEALTHY' ? 'text-success' : 'text-danger'}`;
        } catch (e) {}
    }
    setInterval(update, 3000);
    update();
</script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/api/status')
def api_status():
    latest = get_latest_reading()
    health_state = "HEALTHY"
    try:
        conn = sqlite3.connect(DB_NAME)
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM system_logs ORDER BY id DESC LIMIT 1')
        last_log = cursor.fetchone()
        conn.close()
        if last_log:
            health_state = last_log['health_state']
    except Exception:
        health_state = "FAILED"

    return jsonify({
        "current_reading": latest,
        "health_state": health_state,
        "stale_threshold": STALE_THRESHOLD_SECONDS,
        "alert_threshold": TEMP_HIGH_THRESHOLD
    })

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
''')

# Initialize DB explicitly
from database import init_db
init_db()

print("All Assignment 7.2 files generated successfully!")

All Assignment 7.2 files generated successfully!


In [3]:
%%bash -b
pkill -f "python3 app.py" || true
pkill -f "python3 monitor_engine.py" || true
pkill -f "python3 sensor_generator.py" || true

nohup python3 app.py > flask.log 2>&1 &
nohup python3 monitor_engine.py > monitor.log 2>&1 &
nohup python3 sensor_generator.py > sensor.log 2>&1 &

In [4]:
from google.colab import output
print("🔗 DASHBOARD URL (BEFORE FAILURE):")
print(output.eval_js("google.colab.kernel.proxyPort(5000)"))

🔗 DASHBOARD URL (BEFORE FAILURE):
https://5000-m-s-kkb-use5c0-1p81ucdx91bij-c.us-east5-0.prod.colab.dev


In [5]:
%%bash
# Change database file permissions to READ-ONLY
chmod 444 sensor_history.db
echo "Database permission changed to READ-ONLY (chmod 444)"

Database permission changed to READ-ONLY (chmod 444)


In [6]:
# Read log file showing captured write failures and exception types
with open("app_activity.log", "r") as f:
    logs = f.readlines()
    for log in logs[-15:]:
        print(log.strip())

2026-10-18 22:32:05,538 [INFO] 127.0.0.1 - - [18/Oct/2026 22:32:05] "GET /api/status HTTP/1.1" 200 -
2026-10-18 22:32:08,557 [INFO] 127.0.0.1 - - [18/Oct/2026 22:32:08] "GET /api/status HTTP/1.1" 200 -
2026-10-18 22:32:11,547 [INFO] 127.0.0.1 - - [18/Oct/2026 22:32:11] "GET /api/status HTTP/1.1" 200 -
2026-10-18 22:32:14,558 [INFO] 127.0.0.1 - - [18/Oct/2026 22:32:14] "GET /api/status HTTP/1.1" 200 -
2026-10-18 22:32:17,550 [INFO] 127.0.0.1 - - [18/Oct/2026 22:32:17] "GET /api/status HTTP/1.1" 200 -
2026-10-18 22:32:20,553 [INFO] 127.0.0.1 - - [18/Oct/2026 22:32:20] "GET /api/status HTTP/1.1" 200 -
2026-10-18 22:32:23,549 [INFO] 127.0.0.1 - - [18/Oct/2026 22:32:23] "GET /api/status HTTP/1.1" 200 -
2026-10-18 22:32:25,581 [INFO] 127.0.0.1 - - [18/Oct/2026 22:32:25] "GET /api/status HTTP/1.1" 200 -
2026-10-18 22:32:29,539 [INFO] 127.0.0.1 - - [18/Oct/2026 22:32:29] "GET /api/status HTTP/1.1" 200 -
2026-10-18 22:32:32,545 [INFO] 127.0.0.1 - - [18/Oct/2026 22:32:32] "GET /api/status HTTP/1

In [7]:
%%bash
# Restore WRITE permissions to database file
chmod 664 sensor_history.db

# Restart services
pkill -f "python3 app.py" || true
pkill -f "python3 monitor_engine.py" || true
pkill -f "python3 sensor_generator.py" || true

nohup python3 app.py > flask.log 2>&1 &
nohup python3 monitor_engine.py > monitor.log 2>&1 &
nohup python3 sensor_generator.py > sensor.log 2>&1 &

echo "Database permissions restored and services restarted successfully!"

Database permissions restored and services restarted successfully!


In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("sensor_history.db")
df_readings = pd.read_sql_query("SELECT * FROM sensor_readings ORDER BY id DESC LIMIT 5", conn)
df_logs = pd.read_sql_query("SELECT * FROM system_logs ORDER BY id DESC LIMIT 5", conn)
conn.close()

print("=== RECOVERED SENSOR READINGS ===")
display(df_readings)

print("\n=== RECOVERED SYSTEM LOGS ===")
display(df_logs)

=== RECOVERED SENSOR READINGS ===


,id,timestamp,temperature,humidity,status
0,22,2026-10-18T22:33:29.643291,24.29,48.68,ACCEPTED
1,21,2026-10-18T22:33:24.635566,24.20,49.66,ACCEPTED
2,20,2026-10-18T22:33:24.233314,24.69,47.59,ACCEPTED
3,19,2026-10-18T22:33:19.222301,22.07,48.66,ACCEPTED
4,18,2026-10-18T22:33:14.210290,24.72,46.30,ACCEPTED



=== RECOVERED SYSTEM LOGS ===


,id,timestamp,event_type,message,health_state,component,exception_type
0,4,2026-10-18 22:33:26,STATE_CHANGE,System transitioned to HEALTHY state.,HEALTHY,MONITOR_ENGINE,NONE
1,3,2026-10-18 22:33:24,STARTUP,System startup initiated.,RECOVERING,MONITOR_ENGINE,NONE
2,2,2026-10-18 22:31:50,STATE_CHANGE,System transitioned to HEALTHY state.,HEALTHY,MONITOR_ENGINE,NONE
3,1,2026-10-18 22:31:48,STARTUP,System startup initiated.,RECOVERING,MONITOR_ENGINE,NONE
